In [13]:
import pandas as pd

df = pd.read_parquet("data/processed/notebook2_labeled_table.parquet")
print(df.shape)

(96470, 18)


In [14]:
df = df.sort_values("order_purchase_timestamp").reset_index(drop=True)

print("earliest order:", df["order_purchase_timestamp"].min())
print("latest order:", df["order_purchase_timestamp"].max())

earliest order: 2016-09-15 12:16:38
latest order: 2018-08-29 15:00:37


In [15]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]

print("train:", train.shape)
print("val:", val.shape)
print("test:", test.shape)

train: (67529, 18)
val: (14470, 18)
test: (14471, 18)


إجمالي الصفوف : 96,470

train: 67,529 صف (~70%)
val: 14,470 صف (~15%)
test: 14,471 صف (~15%)

(الأرقام محسوبة من train_end = int(96470×0.70) و val_end = int(96470×0.85))

In [10]:
col = "order_purchase_timestamp"
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, part[col].min(), "→", part[col].max())

train 2016-09-15 12:16:38 → 2018-04-15 20:12:35
val 2018-04-15 20:17:11 → 2018-06-21 08:29:29
test 2018-06-21 08:41:07 → 2018-08-29 15:00:37


earliest order (كامل الجدول): 2016-09-15
latest order (كامل الجدول): 2018-08-29

الأقسام مرتبة زمنيًا بدون تداخل:
train = الأقدم،
val = الأوسط، 
test = الأحدث.

In [16]:
for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name)
    print(part["label"].value_counts(normalize=True))
    print()

train
label
On-time    0.909728
Late       0.090272
Name: proportion, dtype: float64

val
label
On-time    0.946579
Late       0.053421
Name: proportion, dtype: float64

test
label
On-time    0.933868
Late       0.066132
Name: proportion, dtype: float64



### النتائج

* **Train:** On-time 90.97% | Late 9.03%
* **Val:** On-time 94.66% | Late 5.34%
* **Test:** On-time 93.39% | Late 6.61%

### الملاحظة

نسبة الطلبيات المتأخرة (**Late**) مش ثابتة تمامًا بين الأقسام الثلاثة، وبتتراوح بين **~5.3% و~9%**.

هاد فرق متوقع ومنطقي بسبب استخدام **Time-Based Split**؛ بما إن كل قسم بيمثل فترة زمنية مختلفة (`train` = الأقدم، `test` = الأحدث)، فأي تغيّر حقيقي بجودة التوصيل بمرور الوقت، مثل التحسينات اللوجستية أو التأثيرات الموسمية، ممكن ينعكس على نسبة الـ `label` بكل قسم.

### ليش هاد مو مشكلة؟

لو كنا استخدمنا **Random Split**، هاي الفروقات كانت غالبًا رح تقل لأنو كل الفترات بتنخلط ببعض، بس كنا رح نخسر معلومة حقيقية عن تطور جودة التوصيل بمرور الوقت.

لذلك، الاختلاف البسيط بالنسب هون **طبيعي ومتوقع** بسبب اختيار الـ `Time-Based Split`.

### الأثر على الخطوات الجاية

1. تأكيد إضافي إنو لازم نستخدم **F1-score أو مقاييس مشابهة**، مش `Accuracy` وحدها، بـ Notebook 6 لأنو الـ **imbalance متغير** وليس ثابت بكل قسم.
2. نقطة ممكن نستكشفها أكتر بـ **Notebook 4 (EDA)**: هل فيه نمط زمني أو موسمي واضح وراء التغيّر بنسبة التأخير؟
3. لازم نتذكر هالنقطة عند تفسير نتائج الموديل لاحقًا؛ لو الأداء اختلف شوي بين `val` و`test`، ممكن يكون هاد جزء من السبب.


In [18]:
import os
os.makedirs("data/processed", exist_ok=True)

train.to_parquet("data/processed/notebook3_train.parquet", index=False)
val.to_parquet("data/processed/notebook3_val.parquet", index=False)
test.to_parquet("data/processed/notebook3_test.parquet", index=False)

print("Saved ✅")
"""
ليش منحفظ 3 ملفات منفصلة مش ملف واحد؟ 

عشان Notebook 4 (وباقي النوتبوكات) يقدروا يقروا بس الملف يلي محتاجينه (مثلاً train بس لـ EDA)

، بدون ما يضطروا يحملوا كل البيانات ويفلتروها كل مرة من جديد. 

هيك كل نوتبوك بياخد بالضبط الـ artifact يلي يخصه.
"""

Saved ✅


'\nليش منحفظ 3 ملفات منفصلة مش ملف واحد؟ \n\nعشان Notebook 4 (وباقي النوتبوكات) يقدروا يقروا بس الملف يلي محتاجينه (مثلاً train بس لـ EDA)\n\n، بدون ما يضطروا يحملوا كل البيانات ويفلتروها كل مرة من جديد. \n\nهيك كل نوتبوك بياخد بالضبط الـ artifact يلي يخصه.\n'

اخترت الـ **time-based split** لأنه المشكلة يلي عم بحلها (التنبؤ بالتأخير) هي أصلاً مشكلة زمنية — بالواقع الموديل رح يتدرب على طلبيات قديمة ويتنبأ بطلبيات جديدة، وما رح يشوف بيانات مستقبلية وقت التدريب.

لو استخدمت **random split**، كان ممكن يصير عندي طلبيات من نفس اليوم موزعة بين `train` و`test` بنفس الوقت. هاد نوع من "data leakage" الزمني، لأنو الموديل بيكون شاف سياق نفس الفترة أثناء التدريب، وهاد بيخلي تقييمه يبان أحسن من الواقع.

### بالـ time-based split:

1. الموديل بيتدرب على الماضي ويتقيّم على المستقبل، بالضبط متل ما رح يشتغل بالواقع.
2. بيظهر مشاكل حقيقية، متل التغيّر التدريجي بنسبة التأخير بمرور الوقت.
3. التقييم بيصير أصدق وأقرب لسيناريو حقيقي.

### العيب الوحيد

إنو نسبة `Late/On-time` مش متطابقة تمامًا بين الأقسام (9% بالـ `train` مقابل 5.3% بالـ `val` و6.6% بالـ `test`)، بس هاد قرار واعي مني — فضّلت الواقعية الزمنية على التوازن الإحصائي المثالي، ووثقت الفرق كملاحظة بدل ما أعدّله بالقوة.

### باختصار

اخترت هاد النوع من التقسيم بناءً على طبيعة المشكلة نفسها (توقع حدث مستقبلي)، مش كخيار عشوائي أو تقني.
